In [1]:
import openeo
import os
import geopandas as gpd
import rasterio
import glob
from rasterio.mask import mask
from rasterio.merge import merge
from pathlib import Path
import numpy as np
from openeo.processes import quantiles, array_element, if_

import time

In [ ]:
# Remove connection to the backend and remove the refresh token 
#from openeo.rest.auth.config import RefreshTokenStore
#RefreshTokenStore().remove()

In [2]:
connection = openeo.connect(url="openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

## Sentinel-2 mosaic from Sentinel-2 L2A

Create the Sentinel-2 mosaic from L2A data. Product description: 

https://dataspace.copernicus.eu/news/2024-2-27-exploring-new-frontier-sentinel-cloudless-mosaics-copernicus-data-space-ecosystem

https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html#sentinel-2-level-3-quarterly-mosaics

In [3]:
spatial_extent_west = {"west": 610950, "east": 674650, "south": 5143820, "north": 5206380, "crs": "EPSG:32632"} # Western South Tyrol glaciers
spatial_extent_east = {"west": 704100, "east": 747040, "south": 5195880, "north": 5219660, "crs": "EPSG:32632"} # Eastern South Tyrol glaciers

In [4]:
years = [2023]
# 2016, 2017,2018, 2019, 2020, 2021, 2022, 2024, 2025, 2026
base_folder = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/")

### 1) Generate the Sentinel-2 L2A cube

Load all the 10 m resolution bands + SCL

In [ ]:
BANDS = ["B02", "B03", "B04", "B08", "B11", "B12"]
BAD_SCL = [1, 3, 7, 8, 9, 10]   

def load_s2(spatial_extent, temporal_extent):
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=BANDS + ["SCL"],
        max_cloud_cover=50,
    )
    return cube.resample_spatial(projection=32632, resolution=10, method="bilinear")

def cloud_mask(cube):
    scl = cube.band("SCL")
    m = scl == BAD_SCL[0]
    for v in BAD_SCL[1:]:
        m = m | (scl == v)
    return m

def create_s2_mosaic(spatial_extent, temporal_extent):
    cube = load_s2(spatial_extent, temporal_extent)
    mask = cloud_mask(cube)
    return (
        cube.mask(mask)
        .filter_bands(BANDS)
        .reduce_dimension(
            dimension="t",
            reducer=lambda x: array_element(quantiles(x, probabilities=[0.25]), index=0),
        )
    )

In [ ]:
for year in years:
    temporal_extent = [f"{year}-07-01T00:00:00Z", f"{year}-09-30T23:59:59Z"]


    regions = {"west": spatial_extent_west, "east": spatial_extent_east}

    results_folder = base_folder / str(year)
    results_folder.mkdir(parents=True, exist_ok=True)

    jobs = {}

    for name, extent in regions.items():
        result = create_s2_mosaic(extent, temporal_extent).save_result(format="GTiff")
        job = result.create_job(title=f"S2_mosaic_{name}_{year}")
        job.start()
        jobs[name] = job
        print(f"{name} {year} started: {job.job_id}")
        time.sleep(10)

    for name, job in jobs.items():
        while job.status() not in ("finished", "error", "canceled"):
            time.sleep(30)
        if job.status() == "finished":
            out_file = results_folder / f"S2_mosaic_{name}_{year}.tiff"
            job.get_results().download_file(out_file)
            print(f"Downloaded {out_file}")
        else:
            print(f"Job {name} {year} ended with status {job.status()}")

### 6) Calculate indices & apply threshold

In [ ]:
indices_out = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/results/S2/")

for year in years:
    in_folder = base_folder / str(year)
    out_folder = indices_out / str(year)
    out_folder.mkdir(parents=True, exist_ok=True)  

    print("Looking in:", in_folder)
    print("Folder exists:", in_folder.exists())

    for f in sorted(in_folder.glob("S2_mosaic*.tiff")):
        with rasterio.open(f) as src:
            print(f"File: {f.name}, CRS: {src.crs}, Size: {src.width}x{src.height}")

            green = src.read(2).astype(np.float32)
            red   = src.read(3).astype(np.float32)
            nir   = src.read(4).astype(np.float32)
            swir  = src.read(5).astype(np.float32)
            nodata = src.nodata
            profile = src.profile.copy()

        # Valid-pixel mask
        valid = (swir > 0) & (red >= 0) & (green >= 0) & (nir >= 0)
        if nodata is not None:
            valid &= (red != nodata) & (swir != nodata) & (green != nodata) & (nir != nodata)

        with np.errstate(divide="ignore", invalid="ignore"):
            red_swir_ratio = np.where(valid, red / swir, np.nan)
            ndsi = np.where(valid, (green - swir) / (green + swir), np.nan)

        # apply treshold to both (0.4 for ndsi and 4 for ratio)
        ndsi_thresh = ndsi > 0.6
        ratio_thresh = red_swir_ratio > 4

        profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

        outputs = {
            "red_swir_ratio": red_swir_ratio,
            "ndsi": ndsi,
            "ndsi_0.6" : ndsi_thresh,
            "ratio_4": ratio_thresh
        }

        for name, arr in outputs.items():
            with rasterio.open(out_folder / f"{f.stem}_{name}.tiff", "w", **profile) as dst:
                dst.write(arr.astype(np.float32), 1)

Looking in: /mnt/CEPH_PROJECTS/provinzBZ_risk_EO/GlacierOutlines/tests/s2_mosaics/2023
Folder exists: True
File: S2_mosaic_east_2023.tiff, CRS: EPSG:32632, Size: 4294x2378


/tmp/ipykernel_1445129/100037189.py:15: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  green = src.read(2).astype(np.float32)
/tmp/ipykernel_1445129/100037189.py:16: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  red   = src.read(3).astype(np.float32)
/tmp/ipykernel_1445129/100037189.py:17: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  nir   = src.read(4).astype(np.float32)
/tmp/ipykernel_1445129/100037189.py:18: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed

File: S2_mosaic_west_2023.tiff, CRS: EPSG:32632, Size: 6370x6256
